# S9 - Sistemas expertos (Pagarium)

**Objetivo (RA5-a/c):** entender la **anatomía** de un sistema experto y el ciclo *reconocer-resolver-actuar* construyendo un micro-motor, viendo **por qué falla** y comparándolo con `experta` y CLIPS.

> Hilo conductor: **Pagarium**, una pasarela de pagos que decide si aprueba o rechaza una transacción.

## El caso: Pagarium

Pagarium es una **pasarela de pagos** (como Stripe o Redsys). Cada pago trae unos **hechos** y, con reglas, Pagarium decide si **aprobar**, **revisar** o **rechazar**:

| Hecho | Ejemplo |
|---|---|
| `importe` | 200 |
| `antiguedad_meses` | 10 |
| `n_intentos` | 5 |

- importe pequeño y pocos reintentos → **aprobar**
- demasiados reintentos → **rechazar** (puede ser fraude)

### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre (una *variable*) |
| `{ "clave": "valor", ... }` | Un **diccionario**: parejas nombre → valor |
| `[a, b, c]` | Una **lista**: varios valores ordenados |
| `def nombre(...):` | Crea una **función** (bloque de código con nombre) |
| `class Nombre:` | Crea una **clase** (plantilla de objetos) |
| `self` | Dentro de una clase, se refiere "a este objeto" |
| `lambda x: ...` | Una función corta escrita en una sola línea |
| `if ...:` / `for ... in ...:` / `while ...:` | Condición / repetición |
| `print(...)` | Escribe en pantalla |

**No necesitas saber programar de memoria**: lee los comentarios `#` y ejecuta las celdas de arriba a abajo.

## 1. Un micro-motor (lo construimos paso a paso)

Vamos a montarlo **pieza a pieza**, pero **en una sola celda**: en Python una clase no se puede repartir entre celdas (los métodos van indentados dentro de ella). Completa cada `# TODO` y ejecuta. Al final verás **por qué falla**.

### Los 4 pasos (dentro de la misma clase)

1. `__init__` — los tres almacenes.
2. `hecho` — añadir hechos.
3. `regla` — añadir reglas.
4. `run` — el ciclo reconocer-actuar.

> ⚠️ **Importante:** los `# TODO` son el ejercicio. Si ejecutas la celda **sin completarlos** (sin borrar el `pass`), el código dará error al usarlo (p. ej. `AttributeError: 'MicroMotor' object has no attribute 'traza'`). Es lo esperado: rellena los 4 métodos antes del Paso 5. La solución completa está en `sesion09_sistemas_expertos_soluciones.ipynb`.

In [ ]:
class MicroMotor:
    # ---------- PASO 1 · los tres almacenes ----------
    def __init__(self):
        # TODO: crea con self (el punto se lee "de este objeto"):
        #   self.hechos = {}   # diccionario para los hechos
        #   self.reglas = []   # lista para las reglas
        #   self.traza  = []   # lista para el orden de disparo
        pass   # borra esta línea cuando completes el TODO

    # ---------- PASO 2 · añadir hechos ----------
    def hecho(self, **h):        # **h recoge las parejas nombre=valor
        # TODO: añade esas parejas al diccionario self.hechos
        #       (pista: self.hechos.update(h))
        pass

    # ---------- PASO 3 · añadir reglas ----------
    def regla(self, nombre, prioridad, condicion, accion):
        # TODO: añade (nombre, prioridad, condicion, accion) a self.reglas
        #       (pista: self.reglas.append((nombre, prioridad, condicion, accion)))
        pass

    # ---------- PASO 4 · el ciclo reconocer-actuar ----------
    def run(self):
        pendientes = True
        # TODO: mientras pendientes sea True:
        #   pon pendientes = False
        #   recorre con for cada (nombre, prioridad, condicion, accion) de self.reglas:
        #     si nombre ya está en self.traza -> continue
        #     si condicion(self.hechos) se cumple:
        #        apunta nombre en self.traza, ejecuta accion(self.hechos), pon pendientes = True
        #   al salir, devuelve self.hechos
        pass

### Paso 5 · Crear el motor y probarlo

In [ ]:
motor = MicroMotor()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)   # hechos del pago

motor.regla("aprobar_pequena", 10, lambda h: h["importe"] < 500,      # condición: importe < 500
            lambda h: h.update(decision="aprobar"))                  # acción: decide "aprobar"
motor.regla("rechazar_reintentos", 10, lambda h: h["n_intentos"] >= 4,  # condición: reintentos >= 4
            lambda h: h.update(decision="rechazar"))                 # acción: decide "rechazar"

print(motor.run())     # -> {'importe': 200, ..., 'decision': 'rechazar'}  ¡y antes pasó por 'aprobar'!
print(motor.traza)     # -> ['aprobar_pequena', 'rechazar_reintentos']

## 2. Actividad — corregir el conflicto con `salience`

Modifica `run()` para que, cuando varias reglas encajen, se dispare **solo la de mayor prioridad** (eso es `salience`). Comprueba que ya no hay contradicción.

> Pista: en vez de recorrer `for` todas las reglas, agrupa las que encajan y elige la de mayor `prioridad` con `max(...)`.

In [ ]:
# TODO: reescribe run() con resolución de conflictos por prioridad
class MicroMotor2:
    ...

## 3. La misma política con `experta`

Antes de escribir reglas, un minuto para los **decoradores**, porque `@DefFacts` y `@Rule` lo son.

### Los decoradores (`@`) en 2 minutos

Un **decorador** es una función que *envuelve* a otra para modificar su comportamiento, y se aplica con `@nombre`. Ejemplo (adaptado de [El Libro de Python](https://ellibrodepython.com/decoradores-python)):

In [ ]:
def mi_decorador(funcion):        # recibe la función a decorar
    def nueva_funcion(a, b):      # la "envuelve"
        print("Antes de llamar")
        c = funcion(a, b)         # llama a la función original
        print("Después de llamar")
        return c
    return nueva_funcion          # devuelve la función envuelta

@mi_decorador                     # aplica el decorador a suma
def suma(a, b):
    return a + b

print(suma(5, 8))
# Antes de llamar
# 13
# Después de llamar

Con **parámetros** (por eso `@Rule(...)` y `@DefFacts()` llevan paréntesis): el decorador se llama con argumentos y devuelve el decorador de verdad. En `experta`, `@Rule(...)` marca un método como **regla** y `@DefFacts()` marca el método de los **hechos iniciales**.

**Dos casos reales:**

- **Registrar en un fichero (`@log`).** Un decorador con parámetros que escribe el resultado de cada función en un fichero. Fíjate en las **tres funciones anidadas**.

In [ ]:
def log(fichero_log):                     # decorador "de fábrica": recibe el nombre del fichero
    def decorador_log(func):              # el decorador de verdad: recibe la función
        def decorador_funcion(*args, **kwargs):       # *args/**kwargs = "cualquier argumento"
            with open(fichero_log, 'a') as f:         # abre el fichero en modo añadir
                output = func(*args, **kwargs)        # llama a la función original
                f.write(f"{output}\n")               # escribe el resultado en el fichero
        return decorador_funcion
    return decorador_log

@log('resultados.log')
def suma(a, b):
    return a + b

suma(10, 30)      # crea (o añade a) resultados.log la línea "40"

- **Permitir solo a usuarios autenticados.** Un decorador que ejecuta la función solo si la variable `autenticado` es `True`.

In [ ]:
autenticado = True

def requiere_autenticacion(f):
    def funcion_decorada(*args, **kwargs):
        if not autenticado:                          # si no está autenticado
            print("Error. El usuario no se ha autenticado")
        else:
            return f(*args, **kwargs)                # si sí, ejecuta la función
    return funcion_decorada

@requiere_autenticacion
def di_hola():
    print("Hola")

di_hola()          # -> Hola (si autenticado=True) / Error... (si False)

### Lo construimos paso a paso (en una sola celda)

Igual que el micro-motor: la clase va **en una sola celda**. El parche y los `import` van al principio; completa cada `# TODO`.

In [ ]:
%pip install experta

import collections, collections.abc
if not hasattr(collections, "Mapping"):   # parche para Python 3.10+
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *                     # importa Fact, Rule, KnowledgeEngine...

class Pagarium(KnowledgeEngine):          # hereda el motor de reglas
    # ---------- PASO 2 · los hechos iniciales (@DefFacts) ----------
    @DefFacts()
    def inicio(self):
        # TODO: devuelve el hecho inicial accion="evaluar"
        #       (pista: yield Fact(accion="evaluar"))
        pass

    # ---------- PASO 3 · las reglas (@Rule) ----------
    @Rule(Fact(accion="evaluar"), Fact(importe=P(lambda i: i < 500)),
          Fact(n_intentos=P(lambda n: n < 4)), salience=20)   # salience = prioridad
    def aprobar(self):
        # TODO: declara decision="aprobar" con self.declare(Fact(...))
        pass

    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)   # más prioridad
    def rechazar(self):
        # TODO: declara decision="rechazar"
        pass

### Ejecutar el motor

In [ ]:
motor = Pagarium(); motor.reset()      # crea el motor y lo deja listo
motor.declare(Fact(importe=200, n_intentos=5))   # hechos del pago
motor.run()                            # ejecuta el ciclo de inferencia
print([dict(f) for f in motor.facts.values() if "decision" in f])   # solo hechos con 'decision'

## 4. La misma política en CLIPS (`clipspy`)

`clipspy` es el enlace de Python con **CLIPS 6.4** (producción). Las reglas se escriben en una sintaxis LISP con paréntesis, y la prioridad se marca con `(declare (salience 30))`.

In [ ]:
%pip install clipspy
import clips

env = clips.Environment()              # crea el entorno de CLIPS
# build(...) define UNA regla cada vez (no dos a la vez). Cada llamada añade una regla.
env.build("(defrule aprobar (importe ?i&:(< ?i 500)) (n_intentos ?n&:(< ?n 4)) => (assert (decision aprobar)))")
env.build("(defrule rechazar (declare (salience 30)) (n_intentos ?n&:(>= ?n 4)) => (assert (decision rechazar)))")
#   (defrule nombre   condiciones-con-patrones       =>   acciones(assert añade hechos))

env.assert_string("(importe 200)")     # añade un hecho: importe 200
env.assert_string("(n_intentos 5)")    # añade otro hecho
env.run()                              # ejecuta
print([str(f) for f in env.facts()])   # lista todos los hechos finales

### Otro dominio: control de calefacción

El **mismo** micro-motor sirve para cualquier dominio; solo cambian las reglas y los hechos. Aquí controla una calefacción.

In [ ]:
# El mismo micro-motor, otro dominio: calefacción
motor = MicroMotor()
motor.hecho(temperatura=18, hora=23, ventana_abierta=False)   # hechos del ambiente

motor.regla("frio", 10, lambda h: h["temperatura"] < 20, lambda h: h.update(encender=True))   # si hace frío, encender
motor.regla("noche", 10, lambda h: h["hora"] >= 22, lambda h: h.update(modo="eco"))           # si es de noche, modo eco
motor.regla("ventilar", 5, lambda h: h["ventana_abierta"], lambda h: h.update(encender=False))  # ventana abierta: no encender

print(motor.run())    # hechos finales
print(motor.traza)    # reglas disparadas

### Otro dominio: clasificación de animales

`experta` con reglas de un zoólogo. Fíjate en el **encadenamiento hacia delante**: un hecho (`pelo`) dispara una regla que crea otro hecho (`mamifero`), y ese dispara la siguiente.

In [ ]:
class Animales(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(analizar=True)      # arranque

    @Rule(Fact(analizar=True), salience=10)
    def datos(self):                    # carga los datos del animal a clasificar
        self.declare(Fact(pelo=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))

    @Rule(Fact(pelo=True))              # si tiene pelo -> es mamífero (regla intermedia)
    def mamifero(self):
        self.declare(Fact(mamifero=True))

    @Rule(Fact(mamifero=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))
    def guepardo(self):                 # todas las pistas encajan -> conclusión
        print("IDENTIFICADO: guepardo")

engine = Animales(); engine.reset(); engine.run()   # ejecuta: se ve el encadenamiento

### Más reglas en CLIPS

Dos reglas que deciden la tarifa según el importe: a partir de 100 €, tarifa premium.

In [ ]:
env = clips.Environment()
# (?i&:(< ?i 100))  significa: la variable ?i debe cumplir (< ?i 100) = "?i < 100"
env.build("(defrule pequena (importe ?i&:(< ?i 100)) => (assert (tarifa basica)))")
env.build("(defrule grande (importe ?i&:(>= ?i 100)) => (assert (tarifa premium)))")
env.assert_string("(importe 150)")   # importe 150 -> cumple la regla 'grande'
env.run()
print([str(f) for f in env.facts()])

**Cierre:** compara la traza de los tres motores y explica en 3 líneas qué papel juega la **resolución de conflictos**.